# CogLog seven-segment Swift-YOLO retrain (Colab free)

- Trainer: [Seeed-Studio/ModelAssistant](https://github.com/Seeed-Studio/ModelAssistant) (MIT)
- Dataset: [LibreYOLO/digits-t2eg6](https://huggingface.co/datasets/LibreYOLO/digits-t2eg6) (CC BY 4.0)
- Target contract: input **192**, **10** digit classes (0–9), TFLite float32

Upload this notebook to Google Colab, set **Runtime → GPU**, then run all cells.
Details: `tool/seven_segment/colab_retrain.md` in the CogLog repo.

## 0. GPU check

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable Runtime → Change runtime type → GPU (T4)"

## 1. Mount Drive (recommended on free Colab)

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/coglog_seven_segment")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
print(DRIVE_ROOT)

## 2. Fetch CogLog helper scripts

Uses the public CogLog `main` tree. Change `COGLOG_REF` if you need a PR branch.

In [ ]:
import os
from pathlib import Path

COGLOG_REF = os.environ.get("COGLOG_REF", "main")
WORK = Path("/content/coglog_seven_segment")
WORK.mkdir(parents=True, exist_ok=True)
%cd /content
!rm -rf /content/CogLog_sparse
!git clone --depth 1 --filter=blob:none --sparse https://github.com/madahataraku/CogLog.git CogLog_sparse
%cd /content/CogLog_sparse
!git sparse-checkout set tool/seven_segment
import subprocess
subprocess.check_call(["git", "fetch", "--depth", "1", "origin", COGLOG_REF])
subprocess.check_call(["git", "checkout", "FETCH_HEAD"])
TOOLS = Path("/content/CogLog_sparse/tool/seven_segment")
assert (TOOLS / "prepare_digits_t2eg6.py").is_file()
print("helpers", TOOLS, "ref", COGLOG_REF)

## 3. Install ModelAssistant (SSCMA)

In [ ]:
%cd /content
!rm -rf ModelAssistant
!git clone --depth 1 https://github.com/Seeed-Studio/ModelAssistant.git
%cd /content/ModelAssistant
!python -m pip -q install -U pip wheel
!python -m pip -q install -r requirements.txt
!python -m pip -q install pillow huggingface_hub
!python tools/train.py --help | head -n 5

## 4. Download digits-t2eg6 and convert to COCO

In [ ]:
DATA = WORK / "datasets/digits_t2eg6"
!python "{TOOLS}/prepare_digits_t2eg6.py" --work-dir "{DATA}" --symlink-images
COCO = DATA / "coco"
assert (COCO / "train" / "_annotations.coco.json").is_file()
print("coco", COCO)

## 5. Train Swift-YOLO tiny @ 192

Free Colab: start with `EPOCHS=50`. Resume later with `--resume` if the session drops.

In [ ]:
import os
from pathlib import Path

%cd /content/ModelAssistant
EPOCHS = int(os.environ.get("EPOCHS", "50"))
BATCH = int(os.environ.get("BATCH", "16"))
WORK_DIR = Path("/content/ModelAssistant/work_dirs/coglog_digits192")
WORK_DIR.mkdir(parents=True, exist_ok=True)

!python tools/train.py \
  configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
  --work-dir "{WORK_DIR}" \
  --cfg-options \
    num_classes=10 \
    width=192 \
    height=192 \
    batch={BATCH} \
    epochs={EPOCHS} \
    data_root="{COCO}/" \
    train_ann="train/_annotations.coco.json" \
    train_data="train/" \
    val_ann="valid/_annotations.coco.json" \
    val_data="valid/"

## 6. Export TFLite float32

In [ ]:
from pathlib import Path

%cd /content/ModelAssistant
ckpt = Path(WORK_DIR / "last_checkpoint")
assert ckpt.is_file(), ckpt
weights = Path(ckpt.read_text().strip())
print("weights", weights)

!python tools/export.py \
  configs/swift_yolo/swift_yolo_tiny_1xb16_300e_coco.py \
  "{weights}" \
  --format tflite \
  --imgsz 192 192 \
  --cfg-options \
    num_classes=10 \
    width=192 \
    height=192 \
    data_root="{COCO}/" \
    train_ann="train/_annotations.coco.json" \
    train_data="train/" \
    val_ann="valid/_annotations.coco.json" \
    val_data="valid/"

## 7. Copy artifacts to Drive

In [ ]:
import shutil
from pathlib import Path

out = DRIVE_ROOT / "exports"
out.mkdir(parents=True, exist_ok=True)
shutil.copy2(weights, out / weights.name)
for tflite in Path(WORK_DIR).rglob("*.tflite"):
    shutil.copy2(tflite, out / tflite.name)
    print("tflite", tflite)
notice_src = DATA / "NOTICE.md"
if notice_src.is_file():
    shutil.copy2(notice_src, out / "DATASET_NOTICE.md")
print("copied to", out)
print("Next: on macOS run tool/seven_segment/convert_coreml.py with the .pth")